# Reproducing Liu et al. (2026) from dandiset 001750

**Paper.** Liu R-F, Huang M, Shen Y, Shao M, Jing J et al. (2026). An atlas of primate insular
cortex reveals a signal-processing strategy in von Economo neurons. *Nature Cell Biology* 28,
1742-1757. [doi:10.1038/s41556-026-02009-4](https://doi.org/10.1038/s41556-026-02009-4). The
claims file was written from the version of record (Europe PMC, PMC13457102): main text, legends,
Methods; this attempt also had the Supplementary Information PDF, which says nothing about the Na+
recordings.

**Dandiset.** [DANDI:001750](https://dandiarchive.org/dandiset/001750/0.260825.0222), version
0.260825.0222. The paper cites no version. This is the only published version, and the most
recent; the draft has the same 35 assets and size. It is pinned for that reason.

**What the dandiset holds.** 35 NWB files, one per somatic outside-out patch, from one macaque
(Mfa-221206), sessions 221206AS (17 files) and 221206HS (18 files): voltage-clamp currents to
5 mV steps from -80 mV, 1 to 10 repetitions per patch. No cell type and no patch capacitance.

**Outside inputs.** Cell types come from the paper's code table
`ephysio/data_ephy/rawdata_ephysio/metadata_outSideOutSodiumCurrentRec.csv` in
github.com/RFLiu2021/AIC_proj at commit 44cd8a6 (claims `with_labels`). Two claims on other data
(`elsewhere`) are tested on MetaboLights MTBLS13927, metadata revision MTBLS13927_01, and one on
tables of the same repository commit (`ephysio/data_ephy/sumData_StimWM.xlsx` and
`ephysio/data_ephy/VENsubtypeCmp_extFig8ab/VENsubtype_APs.mat` and
`ephysio/data_ephy/PeakIdx_Fig5e.xlsx`).

| Claim | Data | Tested here |
|---|---|---|
| na-cell-table-link | with labels | tested |
| fig4d-sodium-current-lower | dataset (labels from the link) | tested |
| model-na-ratio | dataset | tested (depends on fig4d) |
| na-patch-capacitance | dataset | blocked, no capacitance in the files |
| fig8a-metabolite-gradient, fig8de-glycolytic-bias | elsewhere (MetaboLights) | tested |
| fig4j-eepsp-amplitude | elsewhere (code repository table) | tested |
| ven-subtypes-similar-physiology | elsewhere (code repository tables) | tested |
| fig3e-phase-peak-index | elsewhere (code repository table) | tested |
| fig3c-ais-distance-length | elsewhere (code repository table) | blocked, no slice or animal labels |
| 24 other claims (Patch-seq, connectivity, PSCs, scRNA-seq, morphology, models, eEPSP latency) | elsewhere | blocked, reasons under "Claims not tested here" |

In [ ]:
import io
import re
import time
from collections import Counter
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import AnovaRM, anova_lm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway

import nacurrent as nc

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = nc.list_assets()
patches = nc.load_all()
table = nc.cell_table()
print(f"streamed {len(patches)} files ({assets['size'].sum() / 1e9:.2f} GB) of dandiset {nc.DANDISET} "
      f"version {nc.VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "file": p["file"][24:], "session": p["meta"]["session_id"], "subject": p["meta"]["subject_id"],
    "cell_id": p["meta"]["cell_id"], "location": p["meta"]["location"], "reps": p["nrep"],
    "sweeps": len(p["sweeps"]), "samples": int(p["info"].n.iloc[0]), "rate_Hz": round(p["rate"]),
    "onset_ms": p["onset"] / p["rate"] * 1e3, "step_ms": p["step_ms"],
    "levels_mV": f"{p['levels'].min():.0f} to {p['levels'].max():.0f}",
    "unit": "/".join(sorted(set(p["info"].unit))), "stim_unit": "/".join(sorted(set(p["info"].stim_unit))),
    "series_fields": "/".join(sorted(set(p["info"].fields))),
    "holding_pA": round(float(np.median(p["mean"][:, :p["onset"] - 100])), 1),
    "noise_sd_pA": round(p["rep_sd"], 2)} for p in patches])
print("Files per session:", info.session.value_counts().to_dict(), "; subjects:", info.subject.unique().tolist())
print("Sampling rates (Hz):", sorted(info.rate_Hz.unique()), "; samples per sweep:", sorted(info.samples.unique()))
print("Step onset (ms):", sorted(info.onset_ms.unique()), "; step duration (ms):", sorted(info.step_ms.unique()))
print("Sweeps per file:", info.sweeps.value_counts().to_dict(), "; repetitions per file:", info.reps.value_counts().sort_index().to_dict())
print("Units:", info.unit.unique().tolist(), info.stim_unit.unique().tolist())
print("Datasets in each VoltageClampSeries:", info.series_fields.unique().tolist())
print("General groups of the first file:", patches[0]["meta"]["keys_general"])
print("session_start_time values:", sorted({p["meta"]["session_start"] for p in patches}),
      "; session ids:", sorted({p["meta"]["session_id"] for p in patches}))
# Step levels: sweep s should step to -80 + 5 (s - 1) mV.
dev = max(np.abs(p["levels"] - (-80 + 5 * (p["sweeps"] - 1))).max() for p in patches)
print("Largest deviation of a sweep's step level from -80 + 5 (sweep - 1) mV:", round(float(dev), 3))
print("Command before the step, every file and sweep (mV):",
      sorted({round(float(np.median(p["command"][0][:, :p["onset"] - 1])), 1) for p in patches}))
# Recordings deposited twice: identical traces.
dup = info.assign(digest=[p["digest"] for p in patches]).digest.duplicated(keep=False)
print("Files whose repetition-1 traces are identical to another file's:", int(dup.sum()))
print("\n", info.drop(columns=["series_fields", "unit", "stim_unit", "subject", "location"]).to_string(index=False))
print("\nCurrent range over all files and samples (pA):",
      round(min(p["mean"].min() for p in patches), 1), "to", round(max(p["mean"].max() for p in patches), 1))

All 35 files are from one subject, sampled at 100 kHz (the Methods' rate), with 6500 samples
(65 ms) per sweep and the step from 20 ms lasting 25 ms (the Methods' 25 ms). Sweep s steps to
-80 + 5 (s - 1) mV, so the Methods' -80 to 0 mV are sweeps 1 to 17; files hold 28 sweeps (to
+55 mV) or 34 (to +85 mV). Sweep 1 is the -80 mV "step", with no change of command. The files carry
no capacitance or series resistance compensation values (only `data`, `electrode`, `gain`,
`starting_time` in each series), no cell type, and no intracellular recording tables. The
holding current before the step is far from zero in every file (printed `holding_pA`), so the
traces, though leak subtracted online by the Methods, keep an offset. No two files hold the same
traces. The `starting_time` of the series (0, 1, 2 or 3 s) does not order the sweeps and is not
used. Every file gives the same session start, 2018-01-01 01:00, while the session ids
(221206AS, 221206HS) and the code table's cell IDs (20221206...) date the recordings to
6 December 2022; the start time is a placeholder.

In [ ]:
# The response follows the command: mean current 0.2 to 2 ms after the step onset, minus the 1 ms
# before it, at the steps where Na+ currents peak (-40 to 0 mV) against sweep 1 (no step), and the
# same contrast with the window moved to 10 to 8 ms before the onset (labels shifted in time).
rows = []
for p in patches:
    m, on, r = p["mean"], p["onset"], p["rate"]
    def resp(j, shift=0):
        a, b = on + shift, on + shift
        return m[j, a + int(0.0002 * r):a + int(0.002 * r)].mean() - m[j, b - int(0.001 * r):b].mean()
    act = np.flatnonzero((p["levels"] >= -40) & (p["levels"] <= 0))
    rows.append({"file": p["file"][24:], "step": np.mean([resp(j) for j in act]), "no_step": resp(0),
                 "step_shifted": np.mean([resp(j, -int(0.010 * r)) for j in act])})
resp = pd.DataFrame(rows)
print(resp.describe().loc[["mean", "50%", "min", "max"]].round(2).to_string())
print("Files where the step response is more inward than sweep 1:", int((resp.step < resp.no_step).sum()), "of", len(resp))
print("Wilcoxon, step against no step: p = %.2g; shifted window against no step: p = %.2g" % (
    stats.wilcoxon(resp.step, resp.no_step).pvalue, stats.wilcoxon(resp.step_shifted, resp.no_step).pvalue))

The inward current appears after the command step and not in the same window shifted 10 ms
earlier, so the current and the command share one clock in every file, as far as this contrast
can tell.

## na-cell-table-link

> The dandiset's recordings are the 35 outside-out patches of the paper's code table (14 VEN and
> 21 PC), and the maximal Na+ current computed from each recording agrees with the value the table
> gives for it.

Matched by NWB file name. The maximal inward current is computed under each reading of
fig4d-sodium-current-lower: baseline (a1) as recorded or (a2) minus the mean of the first 100
samples of a cut starting 2 ms before onset; steps (b1) -80 to 0 mV or (b2) all; window (c1)
0 to 20 ms or (c2) 0 to 25 ms after onset. Repetitions are averaged first.

In [ ]:
meas = nc.measure_all(patches)
link = table.merge(meas, left_on="fileName_nwb", right_on="file", how="outer", indicator=True)
print("Table rows:", len(table), "; files:", len(meas), "; matched:", int((link._merge == "both").sum()))
print("Cell types in the table:", table.cellType.value_counts().to_dict())
link["file_cell_id"] = link.file.map({p["file"]: p["meta"]["cell_id"] for p in patches})
# Table cell_ID 20221206AS2c6 against file cell_id Cell_AS006: session letters and cell number.
key = lambda s: (s.str.extract(r"(AS|HS)")[0] + s.str.extract(r"c(\d+)$")[0].astype(int).astype(str))
link["id_agrees"] = key(link.cell_ID) == (link.file_cell_id.str.extract(r"(AS|HS)")[0]
                                          + link.file_cell_id.str.extract(r"(\d+)$")[0].astype(int).astype(str))
print("Table cell_ID agrees with the file's cell_id (session and cell number):", int(link.id_agrees.sum()), "of", len(link))
print("Cell type in the dataset: none (no field holds it), so no type can disagree.")
link["table_pA"] = link.max_Na_current.abs()
cols = ["".join(r) for r in nc.READINGS]
agree = pd.Series({c: int((np.abs(link[c] - link.table_pA) <= 0.1 * link.table_pA).sum()) for c in cols})
ratio = pd.Series({c: float(np.median(link[c] / link.table_pA)) for c in cols})
rho = pd.Series({c: stats.spearmanr(link[c], link.table_pA)[0] for c in cols})
print("\nPer reading: cells within 10% of the table (of 35), median computed/table, Spearman rho")
print(pd.DataFrame({"within_10pct": agree, "median_ratio": ratio.round(2), "spearman": rho.round(3)}).to_string())
print("\nPer cell (pA):")
print(link[["cell_ID", "cellType", "table_pA"] + cols + ["level_mV", "latency_ms"]].round(1).to_string(index=False))

The criterion's readings take the most negative single sample of the averaged trace. Since noise
pushes that minimum down, the next cell runs a variant that is not a reading of the paper and
decides nothing: a moving average over n samples before the peak is taken. It was added after the
readings were run, to see what measure the table's values could come from.

In [ ]:
rows = []
for n in [1, 25, 50, 75, 100, 150]:
    v = nc.measure_all(patches, smooth=n).set_index("file").loc[link.file]
    for c in ["a2b1c1", "a2b1c2"]:
        x = v[c].values
        rows.append({"smooth_samples": n, "smooth_ms": n / 100, "reading": c,
                     "within_10pct": int((np.abs(x - link.table_pA) <= 0.1 * link.table_pA).sum()),
                     "median_ratio": round(float(np.median(x / link.table_pA)), 3),
                     "spearman": round(stats.spearmanr(x, link.table_pA)[0], 3)})
smooth_tab = pd.DataFrame(rows)
print(smooth_tab.to_string(index=False))

Where the maxima of the readings are found, and whether noise could bias one group: per cell,
the repetition count, the noise, the step and latency of the maximum, the I-V curve of the
peaks, and the averaged trace at the step of the maximum.

In [ ]:
# Where the maximum is found, under the code's example reading a2 b1 c1, with each patch's noise
# (SD of the single-repetition traces before the step, averaged over sweeps and repetitions) and
# repetition count. A Na+ current peaks within a few ms of the step and activates above about
# -55 mV; maxima later than 3 ms or at steps below -55 mV are flagged as unlike a Na+ peak. The
# width of the maximum (time below half its value, around it) tells a brief transient of one or a
# few samples from a current that lasts tenths of a ms or more. The widths fall in two groups
# (printed below), and the cut at 0.2 ms lies in the gap between them.
byfile = {p["file"]: p for p in patches}
def peak_width_ms(p, level, latency_ms):
    j = int(np.flatnonzero(np.isclose(p["levels"], level))[0])
    on, rate = p["onset"], p["rate"]
    tr = p["mean"][j]
    seg = tr[on:on + int(0.025 * rate)] - tr[on - int(0.002 * rate):on - int(0.002 * rate) + 100].mean()
    k = int(round(latency_ms * rate / 1e3))
    half = seg[k] / 2
    lo = k
    while lo > 0 and seg[lo - 1] < half:
        lo -= 1
    hi = k
    while hi < len(seg) - 1 and seg[hi + 1] < half:
        hi += 1
    return (hi - lo + 1) / rate * 1e3
link["peak_width_ms"] = [peak_width_ms(byfile[r.file], r.level_mV, r.latency_ms) for r in link.itertuples()]
link["transient"] = link.peak_width_ms < 0.2
link["reps"] = link.file.map(lambda f: byfile[f]["nrep"])
link["noise_sd_pA"] = link.file.map(lambda f: byfile[f]["rep_sd"])
link["unlike_na_peak"] = (link.latency_ms > 3) | (link.level_mV < -55)
print(link[["cell_ID", "cellType", "reps", "noise_sd_pA", "table_pA", "a2b1c1", "level_mV", "latency_ms",
            "peak_width_ms", "transient", "unlike_na_peak"]].round(2).to_string(index=False))
print("\nMaxima unlike a Na+ peak (later than 3 ms or below -55 mV):", int(link.unlike_na_peak.sum()), "of 35;",
      link[link.unlike_na_peak].cellType.value_counts().to_dict())
print("Their computed/table ratio, median:", round(float((link.a2b1c1 / link.table_pA)[link.unlike_na_peak].median()), 2),
      "; the other cells:", round(float((link.a2b1c1 / link.table_pA)[~link.unlike_na_peak].median()), 2))
print("Maxima narrower than 0.2 ms, brief transients:", int(link.transient.sum()), "of 35;",
      link[link.transient].cellType.value_counts().to_dict(), "; also flagged above:", int((link.transient & link.unlike_na_peak).sum()))
print("Widths of the transients (ms):", round(link.peak_width_ms[link.transient].min(), 2), "to", round(link.peak_width_ms[link.transient].max(), 2),
      "; of the other maxima:", round(link.peak_width_ms[~link.transient].min(), 2), "to", round(link.peak_width_ms[~link.transient].max(), 2))
print("Computed/table ratio, median, transients:", round(float((link.a2b1c1 / link.table_pA)[link.transient].median()), 2),
      "; the other cells:", round(float((link.a2b1c1 / link.table_pA)[~link.transient].median()), 2))
print("\nBy cell type (median VEN, median PC, rank-sum p):")
for c in ["reps", "noise_sd_pA"]:
    v, q = link[c][link.cellType == "VEN"], link[c][link.cellType == "PC"]
    print(f"  {c}: {v.median():.2f}, {q.median():.2f}, p = {stats.mannwhitneyu(v, q).pvalue:.2f}")
print("Spearman, noise SD against computed/table ratio (a2b1c1): rho = %.2f, p = %.2g" %
      stats.spearmanr(link.noise_sd_pA, link.a2b1c1 / link.table_pA))

In [ ]:
# Per-cell I-V of the peak inward current (pA) under a2 b1 c1, steps -80 to 0 mV, and the
# averaged trace of each cell at the step of its maximum.
iv = pd.DataFrame({byfile[f]["file"]: nc.iv_curve(byfile[f]["mean"], byfile[f]["levels"], byfile[f]["onset"],
                                                  byfile[f]["rate"]).peak for f in link.file}).T
iv.index = link.cell_ID.values
iv = iv.loc[:, [c for c in iv.columns if c <= 0.5]]
iv.columns = [f"{c:.0f}" for c in iv.columns]
print(iv.round(0).astype(int).to_string())
fig, axs = plt.subplots(5, 7, figsize=(18, 11), sharex=True)
lo_y, hi_y = [], []
for ax, (_, r) in zip(axs.ravel(), link.iterrows()):
    p = byfile[r.file]
    j = int(np.flatnonzero(np.isclose(p["levels"], r.level_mV))[0])
    on, rate = p["onset"], p["rate"]
    seg = p["mean"][j, on - int(0.002 * rate):on + int(0.020 * rate)]
    seg = seg - seg[:100].mean()
    t = (np.arange(len(seg)) - int(0.002 * rate)) / rate * 1e3
    ax.plot(t, seg, lw=0.5, color="k")
    ax.axhline(-r.table_pA, color="tab:blue", lw=0.8)
    ax.plot(r.latency_ms, -r.a2b1c1, "o", color="tab:red", ms=4)
    ax.set_title(f"{r.cell_ID[8:]} {r.cellType} {r.level_mV:.0f} mV", fontsize=8)
    lo_y.append(seg.min()); hi_y.append(seg.max())
for ax in axs[-1]:
    ax.set_xlabel("ms from step onset")
fig.suptitle("Averaged trace at each cell's step of maximum (a2 b1 c1); red, computed maximum; blue, -table value")
plt.tight_layout(); plt.show()
print("Plotted ranges: time -2 to 20 ms; current", round(min(lo_y), 1), "to", round(max(hi_y), 1),
      "pA, each panel on its own axis, no point outside the axes")

**Result: partially reproduced.** All 35 files match the 35 table rows by file name, and each
table cell ID agrees with the file's cell ID (35 of 35). The dataset holds no cell type, so none
disagrees. Under the criterion's readings at most 2 of 35 maxima are within 10% of the table,
below the 28 the criterion requires. The as-recorded readings (a1) are a median 4.15 to 4.61 times
the table values, as the traces keep a holding offset. The baseline-subtracted readings (a2) are a
median 1.32 to 1.40 times the table values, with Spearman 0.47 to 0.68; a2b1c1 agrees best. Under
a2b1c1, 11 of the 35 maxima (6 VEN, 5 PC) lie later than 3 ms after the onset or at steps below
-55 mV, where a Na+ current does not peak. In 10 cells (6 VEN, 4 PC; 9 of them among the 11) the
maximum is a brief biphasic transient 0.07 to 0.16 ms wide, against 0.33 ms or more for every
other maximum, as the traces above show; these are artefacts, not Na+ currents, and their ratio
to the table is a median 1.94 against 1.28 for the other cells. Noise
does not differ by type (median SD 6.9 pA in VENs against 9.6 pA in PCs, p = 0.34), nor does the
number of repetitions (2.5 against 3, p = 0.89). Fixed Bessel filters and a 0 to 3 ms window
(variants below) bring at most 3 of 35 within 10%. The smoothing gives 19 of 35 at 0.5 ms and at
most 26 of 35 at 0.75 ms (median ratio 1.006, Spearman 0.982), below the 28 the criterion
requires, and that width was chosen as the one that fits the table best. So no measure tried
here matches the table to the criterion's tolerance; the smoothed peak comes closest.

In [ ]:
ven, pc = link.cellType == "VEN", link.cellType == "PC"
def compare(x):
    return {"VEN_n": int(ven.sum()), "PC_n": int(pc.sum()), "VEN_median": np.median(x[ven]), "PC_median": np.median(x[pc]),
            "ranksum_p": stats.mannwhitneyu(x[ven], x[pc], alternative="two-sided").pvalue,
            "ttest_p": stats.ttest_ind(x[ven], x[pc]).pvalue,
            "VEN_lower": bool(np.median(x[ven]) < np.median(x[pc]))}
res = pd.DataFrame({c: compare(link[c].values) for c in cols}).T
res.loc["code table (paper's values)"] = compare(link.table_pA.values)
sm75 = nc.measure_all(patches, smooth=75).set_index("file").loc[link.file]
res.loc["variant: a2b1c1 smoothed 0.75 ms"] = compare(sm75["a2b1c1"].values)
print(res.to_string(float_format=lambda v: f"{v:.4g}"))
print("\nReadings with VEN lower and p < 0.05:", [c for c in cols if res.loc[c, "VEN_lower"] and res.loc[c, "ranksum_p"] < 0.05])
print("\nBy session, VEN median, PC median (pA), n VEN, n PC, rank-sum p:")
for s, g in link.assign(session=link.cell_ID.str[8:10]).groupby("session"):
    gv, gp = g.cellType == "VEN", g.cellType == "PC"
    for c in ["a2b1c1", "a1b1c1", "table_pA"]:
        print(f"  {s} {c}: {g[c][gv].median():.1f}, {g[c][gp].median():.1f}, {int(gv.sum())}, {int(gp.sum())}, "
              f"p = {stats.mannwhitneyu(g[c][gv], g[c][gp]).pvalue:.3g}")
# The brief transients of the na-cell-table-link section, by session, and the session contrast
# under a2 b1 c1 without the cells whose maximum is a transient.
print("\nTransient maxima under a2 b1 c1 (n VEN, n PC) by session:")
for s, g in link.assign(session=link.cell_ID.str[8:10]).groupby("session"):
    gv, gp = g.cellType == "VEN", g.cellType == "PC"
    print(f"  {s}: {int((g.transient & gv).sum())} of {int(gv.sum())} VEN, {int((g.transient & gp).sum())} of {int(gp.sum())} PC;",
          "cells", ", ".join(g.cell_ID[g.transient]))
    k = ~g.transient
    print(f"    without them, a2b1c1: VEN median {g.a2b1c1[gv & k].median():.1f}, PC median {g.a2b1c1[gp & k].median():.1f}, "
          f"n {int((gv & k).sum())}, {int((gp & k).sum())}, p = {stats.mannwhitneyu(g.a2b1c1[gv & k], g.a2b1c1[gp & k]).pvalue:.3g}")
    print(f"    the transient cells' a2b1c1 against the table: " + ", ".join(
        f"{r.cell_ID} {r.cellType} {r.a2b1c1:.1f}/{r.table_pA:.1f}" for r in g[g.transient].itertuples()))
print("Current density cannot be computed: no patch capacitance in the files (see na-patch-capacitance).")

The criterion's readings take one sample of an unfiltered 100 kHz trace. The next cell runs
variants that are not readings of the paper and decide nothing: a 4-pole Bessel low-pass, run
once forward as an amplifier's filter acts, at cut-offs fixed in advance, and a window cut to the first 3 ms, where a Na+ current peaks. Unlike
the smoothing above, they were not fitted to the table.

In [ ]:
# Non-deciding variants of the measure that the criterion's readings do not include: a 4-pole
# Bessel low-pass run once forward, as an amplifier's filter acts (-3 dB at the cut-off), at fixed
# cut-offs (2.9, 5, 10 kHz, chosen before their results were seen)
# and a window cut to 0 to 3 ms after onset, alone and together. Per reading: agreement with the
# table (cells within 10%, median ratio, Spearman), the fig4d test and the model-na-ratio ratios.
rows = []
for hz in [None, 2900, 5000, 10000]:
    for w in [None, 3]:
        m = nc.measure_all(patches, bessel_hz=hz, win_ms=w).set_index("file").loc[link.file]
        for c in cols:
            if w and c.endswith("c2"):
                continue  # with the window cut to 3 ms, c1 and c2 are the same
            x = m[c].values
            rows.append({"bessel_kHz": hz / 1e3 if hz else "-", "window": f"0-{w} ms" if w else c[-2:], "reading": c[:4],
                         "within_10pct": int((np.abs(x - link.table_pA) <= 0.1 * link.table_pA).sum()),
                         "median_ratio_to_table": np.median(x / link.table_pA), "spearman": stats.spearmanr(x, link.table_pA)[0],
                         "VEN_median": np.median(x[ven]), "PC_median": np.median(x[pc]),
                         "ranksum_p": stats.mannwhitneyu(x[ven], x[pc], alternative="two-sided").pvalue,
                         "ratio_mean": x[ven].mean() / x[pc].mean(), "ratio_median": np.median(x[ven]) / np.median(x[pc])})
var_tab = pd.DataFrame(rows)
print(var_tab.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
a2 = var_tab[var_tab.reading.str.startswith("a2")]
print("\nBaseline-subtracted (a2) variants with p < 0.05:", int((a2.ranksum_p < 0.05).sum()), "of", len(a2),
      "; with a ratio in 0.619 to 0.756 under some summary:",
      int(((a2.ratio_mean.between(0.619, 0.756)) | (a2.ratio_median.between(0.619, 0.756))).sum()), "of", len(a2))
print("Most cells within 10% of the table under any of these variants:", int(var_tab.within_10pct.max()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for i, (c, lab) in enumerate([("a2b1c1", "reading a2 b1 c1"), ("table_pA", "code table")]):
    ax[i].boxplot([link[c][ven], link[c][pc]], tick_labels=["VEN", "PC"])
    ax[i].scatter(np.r_[np.ones(ven.sum()), 2 * np.ones(pc.sum())], np.r_[link[c][ven], link[c][pc]], s=12, alpha=.6)
    ax[i].set_ylabel("max inward current (pA)"); ax[i].set_title(lab)
plt.tight_layout(); plt.show()
print("Plotted ranges (pA): a2b1c1", round(link.a2b1c1.min(), 1), "to", round(link.a2b1c1.max(), 1),
      "; table", round(link.table_pA.min(), 1), "to", round(link.table_pA.max(), 1), "; no point outside the axes")

**Result: blocked (methods underspecified).** The counts are 14 VEN and 21 PC, as the paper
states, and the VEN median is lower under all eight readings. But the readings disagree on
significance: a1b1c1 (p = 0.045) and a1b1c2 (p = 0.025) find it, and the six others do not
(p = 0.077 to 0.29), so by the criterion the claim is blocked. Both significant readings keep
the holding current of each patch in the measure (a1 values are about four times the table's),
but the other two a1 readings, over all steps, do not find it (a1b2c1 p = 0.27, a1b2c2
p = 0.18).
Under the code's example reading (a2 b1 c1) the medians are 30.4 against 38.1 pA (p = 0.10).

By session under a2 b1 c1, VENs are lower in session HS (15.9 against 40.3 pA, p = 0.0028) and
not in AS (40.7 against 36.2 pA, p = 0.60). The code table's own values show the same pattern
(HS 12.4 against 31.5 pA, p = 0.0028; AS 17.5 against 29.1 pA, p = 0.11). In the paper's values
VENs are lower in both sessions (AS ratio of medians 0.60), and only the significance rests on
HS, with 7 and 10 cells in AS. The reversal in AS under a2 b1 c1 is not a session effect. Under
that reading the maximum is a brief transient (na-cell-table-link section) in 9 of the 17 AS
cells (5 of its 7 VENs, 4 of its 10 PCs) and in 1 of the 18 HS cells (HS1c18). In three of the
AS VENs the transient is 3 to 5 times the table's value (AS4c14 70.7 against 15.5 pA, AS3c11
87.7 against 17.5, AS1c1 42.1 against 13.8). Without the transient cells AS keeps 2 VENs and
6 PCs (37.2 against 34.3 pA, p = 0.86), too few to tell, while HS keeps its difference (15.6
against 40.3 pA, p = 0.0019). The AS result under the readings is thus mostly the artefact.

The criterion's readings take the single most negative sample of an unfiltered 100 kHz trace,
and under a2 b1 c1, the reading for which widths were computed, that sample is a brief
transient in 6 of the 14 VENs and 4 of the 21 PCs, which raises VEN values more than PC values.
Under variants that are not readings of the paper and decide nothing, the difference is
significant in 13 of the 24 baseline-subtracted variants: under every a2 reading with the
2.9 kHz Bessel filter, every one with the window cut to 0 to 3 ms (with any filter or none), and
a2b1c2 with the 5 kHz filter. The code table's values give 14.9 against 29.3 pA (p = 0.001), and the smoothed variant
16.0 against 29.9 pA (p = 0.0008), but that variant was fitted to the table and is not
independent of it. The outcome turns on how the peak is measured, which the paper does not say;
whether the criterion should include a filtered peak is asked of a reader
(`questions/001750.yaml`).

In [ ]:
lo, hi = 0.6875 - 0.069, 0.6875 + 0.069
rows = []
for c in cols + ["table_pA"]:
    x = link[c].values
    r_mean, r_med = x[ven].mean() / x[pc].mean(), np.median(x[ven]) / np.median(x[pc])
    rows.append({"measure": c, "mean_ratio": r_mean, "median_ratio": r_med,
                 "mean_in": lo <= r_mean <= hi, "median_in": lo <= r_med <= hi})
x = sm75["a2b1c1"].values
rows.append({"measure": "variant smoothed 0.75 ms", "mean_ratio": x[ven].mean() / x[pc].mean(),
             "median_ratio": np.median(x[ven]) / np.median(x[pc])})
ratios = pd.DataFrame(rows)
print(ratios.round(3).to_string(index=False))
print("Readings of the measure with a ratio in tolerance under some summary:",
      [r.measure for r in ratios.itertuples() if r.measure in cols and (r.mean_in or r.median_in)])

**Result: blocked (methods underspecified).** The claim depends on fig4d-sodium-current-lower,
which is blocked, and the protocol blocks a claim that depends on a blocked claim. Under the eight
readings the ratios printed above are 0.68 to 0.99. A ratio falls within 0.619 to 0.756 under some
summary only for the four as-recorded readings (a1), none of the baseline-subtracted ones (0.79 to
0.99); the a1 ratios include each patch's holding current. The code table's values give 0.58
(mean) and 0.51 (median), and the smoothed variant 0.58 and 0.54; none of these is in tolerance.
In the filtered and short-window variants (fig4d section), 10 of the 24 baseline-subtracted
variants give a ratio in tolerance under some summary. The Extended Data Fig. 7c legend says the
conductance range was also set from previous publications.

In [ ]:
fields = sorted(set("/".join(sorted(set(p["info"].fields))) for p in patches))
print("Datasets in the VoltageClampSeries of all files:", fields)
# Every value and attribute in every file outside the data series (general, including
# intracellular_ephys/IntracellularElectrode, lab_meta_data, notes, and the series' own attributes).
texts = pd.DataFrame([{"file": p["file"][24:], "path": k, "value": v} for p in patches for k, v in p["meta"]["text"].items()])
print("Entries per file:", texts.groupby("file").size().unique().tolist())
print("Paths present (with the distinct values over the 35 files, at most 3 shown):")
for k, g in texts.groupby("path"):
    vals = sorted(g.value.unique())
    print(f"  {k}: {len(vals)} distinct; " + " | ".join(v[:70] for v in vals[:3]))
print("lab_meta_data or notes in any file:", bool(texts.path.str.contains("lab_meta_data|notes").any()))
cap_words = r"capacit|resist|seal|compens|c[-_ ]?slow|c[-_ ]?fast|r[-_ ]?series|access|farad|ohm"
hits = texts[texts.value.str.contains(cap_words, case=False) | texts.path.str.contains(cap_words, case=False)]
print("Entries naming capacitance, resistance, seal, compensation, farad or ohm:", len(hits))
print(hits.drop_duplicates(["path", "value"]).to_string(index=False) if len(hits) else "")
meta = [nc.asset_metadata(a) for a in assets.asset_id]
print("Asset metadata keys (first asset):", sorted(meta[0]))
print("Assets whose metadata names capacitance or resistance:",
      sum(bool(re.search(r"capacit|resist", str(m), re.I)) for m in meta), "of", len(meta))

**Result: blocked (data missing).** Every file holds the same 52 values and attributes outside
the data series, printed above: the IntracellularElectrode has only a cell ID, a description
("custom-made glass intracellular electrode") and a location (Soma), there is no lab_meta_data and
no notes, and the series' attributes are a description, comments ("no comments") and units. None
names a capacitance, resistance, seal or compensation, and neither does the metadata of any of the
35 assets. The criterion blocks the claim in that case. The Methods say the traces were leak subtracted
online, which would also remove the capacitive transients from which a capacitance might be
estimated; that is inferred from the Methods, and this attempt did not try such an estimate.

In [ ]:
MTBLS = "https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/MTBLS13927/METADATA_REVISIONS/MTBLS13927_01/"
maf = pd.read_csv(io.StringIO(requests.get(MTBLS + "m_MTBLS13927_LC-MS_alternating_hilic_metabolite_profiling_v2_maf.tsv", timeout=120).text), sep="\t")
samples = pd.read_csv(io.StringIO(requests.get(MTBLS + "s_MTBLS13927.txt", timeout=120).text), sep="\t")
scols = [c for c in maf.columns if c[:3] in ("A1C", "AIC", "PrC", "V1-")]
region = {c: "AIC" if c[:3] in ("A1C", "AIC") else ("PrC" if c.startswith("PrC") else "V1") for c in scols}
print("Metabolites:", len(maf), "; sample columns:", len(scols), pd.Series(region).value_counts().to_dict())
print("Sample table names:", samples["Source Name"].tolist())
print("Column 'A1C-1-Q' is read as AIC-1-Q (the sample table spells it so too; the other AIC samples are AIC-).")
X = maf[scols].apply(pd.to_numeric, errors="coerce")
print("Missing values:", int(X.isna().sum().sum()), "; zeros:", int((X == 0).sum().sum()), "; range", float(X.min().min()), "to", float(X.max().max()))
print("Metabolites without a name:", int(maf.metabolite_identification.isna().sum()),
      "; duplicate names:", int(maf.metabolite_identification.duplicated().sum()))

def games_howell(a, b):
    na, nb, va, vb = len(a), len(b), a.var(ddof=1), b.var(ddof=1)
    se = np.sqrt(va / na + vb / nb)
    if se == 0:
        return np.nan
    df = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    q = abs(a.mean() - b.mean()) / se * np.sqrt(2)
    return stats.studentized_range.sf(q, 3, df)

def metab_tests(M):
    out = []
    for i in range(len(M)):
        g = {r: M.iloc[i][[c for c in scols if region[c] == r]].values.astype(float) for r in ("AIC", "PrC", "V1")}
        w = anova_oneway([g["AIC"], g["PrC"], g["V1"]], use_var="unequal").pvalue
        out.append({"welch_p": w, "gh_AIC_V1": games_howell(g["AIC"], g["V1"]), "gh_AIC_PrC": games_howell(g["AIC"], g["PrC"]),
                    "AIC": g["AIC"].mean(), "PrC": g["PrC"].mean(), "V1": g["V1"].mean()})
    return pd.DataFrame(out, index=maf.metabolite_identification)

tests = {"as deposited": metab_tests(X), "log2 (variant)": metab_tests(np.log2(X.clip(lower=1e-9)))}
rows = []
for name, t in tests.items():
    gate = t.welch_p < 0.05
    rows.append({"values": name, "welch_p<0.05": int(gate.sum()),
                 "AIC-V1 (Welch then GH)": int((gate & (t.gh_AIC_V1 < 0.05)).sum()),
                 "AIC-PrC (Welch then GH)": int((gate & (t.gh_AIC_PrC < 0.05)).sum()),
                 "AIC-V1 (GH alone)": int((t.gh_AIC_V1 < 0.05).sum()),
                 "AIC-PrC (GH alone)": int((t.gh_AIC_PrC < 0.05).sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("Expected 228 (205.2 to 250.8) AIC-V1 and 108 (97.2 to 118.8) AIC-PrC, of 581 metabolites.")
zero = X.eq(0).any(axis=1)
print("Metabolites with a zero value:", maf.metabolite_identification[zero].tolist(),
      "; log2 counts without them, AIC-V1 and AIC-PrC (Welch then GH):",
      int((~zero & (tests["log2 (variant)"].welch_p.values < 0.05) & (tests["log2 (variant)"].gh_AIC_V1.values < 0.05)).sum()),
      int((~zero & (tests["log2 (variant)"].welch_p.values < 0.05) & (tests["log2 (variant)"].gh_AIC_PrC.values < 0.05)).sum()))

**Result: not reproduced.** Welch's ANOVA followed by Games-Howell gives 175 metabolites that
differ between AIC and V1 (expected 205.2 to 250.8) and 84 between AIC and PrC (expected 97.2 to
118.8). The Games-Howell test alone gives 179 and 85, and on log2 values 190 and 95 (194 and 98
without the ANOVA gate), all outside tolerance. The single zero value (L-Cystine) does not move the
log2 counts (190 and 95 without that metabolite). The direction holds, with more differences
against V1 than against PrC under every variant. The deposited table holds 558 metabolites, not
the 581 the Results report, which may account for part of the shortfall.

## fig8de-glycolytic-bias (elsewhere: MetaboLights MTBLS13927)

> Compared with V1, AIC shows elevated citrate, reduced 2-ketobutyrate and increased glutamate,
> read as a glycolytic bias.

Welch's ANOVA followed by Games-Howell, AIC against V1. The metabolites are found by name in the
deposited table; every candidate name is printed.

In [ ]:
names = maf.metabolite_identification.fillna("")
cand = {"citrate": names.str.contains(r"citr(?:ate|ic)", case=False) & ~names.str.contains("iso|methyl", case=False),
        "isocitrate (shown only)": names.str.contains("isocitr", case=False),
        "2-ketobutyrate": names.str.contains(r"2-ketobutyr|2-oxobutan|alpha-ketobutyr", case=False),
        "glutamate": names.str.contains(r"^(?:L-)?glutam(?:ate|ic acid)", case=False)}
t = tests["as deposited"]; tl = tests["log2 (variant)"]
rows = []
for k, m in cand.items():
    for i in np.flatnonzero(m.values):
        rows.append({"target": k, "name": names.iloc[i], "AIC_mean": t.AIC.iloc[i], "V1_mean": t.V1.iloc[i],
                     "AIC/V1": t.AIC.iloc[i] / t.V1.iloc[i], "welch_p": t.welch_p.iloc[i], "gh_AIC_V1": t.gh_AIC_V1.iloc[i],
                     "gh_AIC_V1_log2": tl.gh_AIC_V1.iloc[i]})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Names containing 'glutam' (all):", sorted(n for n in names if "glutam" in n.lower()))
print("Names containing 'citr' or 'butyr' (all):", sorted(n for n in names if "citr" in n.lower() or "butyr" in n.lower()))
# The three entries tested. Glutamate is taken as "L-Glutamic acid", the entry named as the amino
# acid itself; "Glutamic acid glutamate" is a second, ambiguously named feature, shown as a variant.
chosen = {"citrate": "Citric acid", "2-ketobutyrate": "2-Ketobutyric acid", "glutamate": "L-Glutamic acid",
          "glutamate (variant)": "Glutamic acid glutamate"}
want = {"citrate": "higher", "2-ketobutyrate": "lower", "glutamate": "higher", "glutamate (variant)": "higher"}
rows = []
for k, n in chosen.items():
    i = int(np.flatnonzero(names.values == n)[0])
    ratio = t.AIC.iloc[i] / t.V1.iloc[i]
    ok = (ratio > 1) == (want[k] == "higher") and t.welch_p.iloc[i] < 0.05 and t.gh_AIC_V1.iloc[i] < 0.05
    rows.append({"target": k, "entry": n, "expected": want[k], "AIC/V1": ratio, "welch_p": t.welch_p.iloc[i],
                 "gh_AIC_V1": t.gh_AIC_V1.iloc[i], "holds_at_0.05": ok})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: partially reproduced.** Citric acid is higher in AIC than in V1 (ratio 1.37,
Games-Howell p = 0.00046), and L-glutamic acid too (ratio 1.35, p = 0.0001). 2-Ketobutyric acid is
lower (ratio 0.37) but not significantly (Games-Howell p = 0.18, Welch's ANOVA p = 0.17; on log2
values p = 0.086). Two of the three parts hold. Glutamate is taken as "L-Glutamic acid"; the
entry "Glutamic acid glutamate" is higher (1.18) without significance (p = 0.15), and had it been
taken, only citrate would hold.

In [ ]:
raw = pd.read_excel(io.BytesIO(nc.repo_file("ephysio/data_ephy/sumData_StimWM.xlsx")), sheet_name="re-orgnized_data", header=None)
block = raw.iloc[1:, 0:4]
block.columns = ["pair", "stim", "PC", "VEN"]
rows, pair, level = [], None, 0
for _, r in block.iterrows():
    if pd.isna(r.PC) and pd.isna(r.VEN):
        pair, level = None, 0
        continue
    if pair is None:
        pair = r.pair if isinstance(r.pair, str) else f"block{len(set(x['pair'] for x in rows)) + 1}"
    level += 1
    rows.append({"pair": pair, "level": level, "stim": r.stim, "PC": float(r.PC), "VEN": float(r.VEN)})
ep = pd.DataFrame(rows)
print("Pairs:", ep.pair.nunique(), "; rows per pair:", ep.groupby("pair").size().value_counts().to_dict())
print("Stimulation values given:", sorted(ep.stim.dropna().unique().tolist()), "; rows without a value:", int(ep.stim.isna().sum()))
print("Amplitude range (unit not stated in the sheet): PC", ep.PC.min(), "to", ep.PC.max(), "; VEN", ep.VEN.min(), "to", ep.VEN.max())
long = ep.melt(id_vars=["pair", "level"], value_vars=["PC", "VEN"], var_name="cell", value_name="amp")
print("\nMean amplitude by level and cell type:")
print(long.pivot_table(index="level", columns="cell", values="amp", aggfunc=["mean", "median", "count"]).to_string())
fit = smf.ols("amp ~ C(cell) * C(level)", data=long).fit()
print("\n(a) two-way ANOVA (type II):")
print(anova_lm(fit, typ=2).to_string(float_format=lambda v: f"{v:.4g}"))
full = long.groupby("pair").filter(lambda g: len(g) == 6)
print(f"\n(b) repeated-measures ANOVA, {full.pair.nunique()} pairs with all three levels:")
print(AnovaRM(full, "amp", "pair", within=["cell", "level"]).fit().anova_table.to_string(float_format=lambda v: f"{v:.4g}"))
d = ep.groupby("pair")[["VEN", "PC"]].mean()
print("\nPairs where the VEN mean amplitude exceeds the PC's:", int((d.VEN > d.PC).sum()), "of", len(d),
      "; Wilcoxon signed-rank p = %.3g" % stats.wilcoxon(d.VEN, d.PC).pvalue)

In [ ]:
# Cross-check with the same sheet's "re-orgnized data" block (columns G to J), which gives the
# stimulation value on every row and a VEN subtype (VENL or VENS) in column J, but no pair ID.
ro = raw.iloc[1:, 6:10].copy()
ro.columns = ["stim", "PC", "VEN", "subtype"]
ro = ro[pd.to_numeric(ro.stim, errors="coerce").notna() & pd.to_numeric(ro.PC, errors="coerce").notna()]
ro = ro.astype({"stim": float, "PC": float, "VEN": float})
print("Re-organized block: rows per stimulation value", ro.stim.value_counts().sort_index().to_dict(),
      "; subtypes", ro.subtype.value_counts(dropna=False).to_dict())
lv = {0.5: 1, 1.0: 2, 2.0: 3}
for s, l in lv.items():
    a = Counter(zip(ep[ep.level == l].PC.round(6), ep[ep.level == l].VEN.round(6)))
    b = Counter(zip(ro[ro.stim == s].PC.round(6), ro[ro.stim == s].VEN.round(6)))
    print(f"stim {s}: pairs in both blocks {sum((a & b).values())}; only in 'copied data' {sorted((a - b).elements())}; "
          f"only in 're-orgnized' {sorted((b - a).elements())}")
rlong = ro.assign(level=ro.stim.map(lv)).melt(id_vars=["level"], value_vars=["PC", "VEN"], var_name="cell", value_name="amp")
print("\nMean amplitude by level and cell type, re-organized block:")
print(rlong.pivot_table(index="level", columns="cell", values="amp", aggfunc=["mean", "count"]).to_string())
print("\nTwo-way ANOVA (type II) on the re-organized block (no pair ID, so no repeated-measures form):")
print(anova_lm(smf.ols("amp ~ C(cell) * C(level)", data=rlong).fit(), typ=2).to_string(float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** In the 26 pairs of the "copied data" block, the VEN amplitudes are higher
at every stimulation level (mean 0.0067, 0.0133 and 0.0193 against 0.0036, 0.0066 and 0.0128),
and in 21 of 26 pairs. The cell-type effect is significant under both readings: the ordinary
two-way ANOVA gives F = 7.62, p = 0.0065, and the repeated-measures ANOVA F = 8.72, p = 0.0068.
The "re-orgnized data" block of the same sheet, which gives the stimulation value on every row,
shares 69 of its 78 rows with the copied block (24, 23 and 22 at the three levels) and differs in
the other 9; its two-way ANOVA gives p = 0.0033 for cell type, against the paper's P = 0.0034, so
the paper's test was most likely run on that block. VENs are higher there too at every level. The
stimulation level effect has p = 5.3e-05 and 1.6e-08 (copied block) and 7.8e-05 (re-organized).
The table gives stimulation values of 0.5, 1 and 2 (unit "mA") where the Methods give 0.05 to
0.2 mA, and no unit for the amplitudes.

## ven-subtypes-similar-physiology (elsewhere: code repository tables)

> VEN-L and VEN-S have similar physiological properties.

The criterion judges three tests: rank-sum with Benjamini-Hochberg FDR on AP width and on AP
amplitude of 85 VEN-L and 41 VEN-S (Extended Data Fig. 9c,d), and the two-way ANOVA of eEPSP
amplitudes between 12 VEN-L and 12 VEN-S (Extended Data Fig. 9f legend). The claims file points
to the recordings of DANDI:001746; the paper's code repository holds the per-cell AP values of
Extended Data Fig. 9c,d and the subtype of each eEPSP pair (column J of the eEPSP sheet), and
these are used. The FDR family is not stated; since BH q is never below the raw p, it cannot
turn a non-significant test significant.

In [ ]:
# Extended Data Fig. 9e,f: 12 VEN-L and 12 VEN-S paired with PCs, two VENs of unidentified subtype
# excluded; "p = 0.9852 with two-way ANOVA between VEN-L and VEN-S". Subtypes from column J of the
# re-organized block (cell above); rows without a subtype are the excluded VENs.
sub = ro.dropna(subset=["subtype"]).assign(level=lambda d: d.stim.map(lv))
print("VENs per subtype and level:", sub.groupby(["subtype", "level"]).size().to_dict())
print("Rows without a subtype:", int(ro.subtype.isna().sum()), "(two per level)")
print("Mean VEN amplitude by level:\n", sub.pivot_table(index="level", columns="subtype", values="VEN", aggfunc="mean").to_string())
print("\nVEN-L against VEN-S, two-way ANOVA (subtype x level, type II):")
print(anova_lm(smf.ols("VEN ~ C(subtype) * C(level)", data=sub).fit(), typ=2).to_string(float_format=lambda v: f"{v:.4g}"))
for s, paper_p in [("VENL", 0.05), ("VENS", 0.0355)]:
    d = sub[sub.subtype == s].melt(id_vars=["level"], value_vars=["PC", "VEN"], var_name="cell", value_name="amp")
    p = anova_lm(smf.ols("amp ~ C(cell) * C(level)", data=d).fit(), typ=2).loc["C(cell)", "PR(>F)"]
    print(f"Context, {s} against its paired PCs, cell-type effect p = {p:.4f} (legend p = {paper_p})")

In [ ]:
# Extended Data Fig. 9c,d: AP width and amplitude of 85 VEN-L and 41 VEN-S, per-cell values from the
# code repository (VENsubtypeCmp_extFig8ab/VENsubtype_APs.mat, read by extFig9ad_plotVENsubtypeAP_Cmp.m).
import scipy.io as sio
aps = sio.loadmat(io.BytesIO(nc.repo_file("ephysio/data_ephy/VENsubtypeCmp_extFig8ab/VENsubtype_APs.mat")))
vals = {g: {k: aps[g][0, 0][k].ravel().astype(float) for k in ("spike_halfWidth", "AP_Amplitude_mV")} for g in ("VENL", "VENS")}
raw_p = []
for k in ("spike_halfWidth", "AP_Amplitude_mV"):
    a, b = vals["VENL"][k], vals["VENS"][k]
    raw_p.append(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue)
    print(f"{k}: VEN-L n = {len(a)}, median {np.median(a):.2f}, range {a.min():.2f} to {a.max():.2f}; "
          f"VEN-S n = {len(b)}, median {np.median(b):.2f}, range {b.min():.2f} to {b.max():.2f}; rank-sum p = {raw_p[-1]:.3f}")
q = multipletests(raw_p, method="fdr_bh")[1]
print("Benjamini-Hochberg over the two tests: q =", np.round(q, 3).tolist(),
      "; BH q is never below the raw p, so no larger family of features can make either significant.")

**Result: reproduced.** None of the three tests is significant. AP width (half-width) is a median
3.28 ms in 85 VEN-L and 3.30 ms in 41 VEN-S (rank-sum p = 0.55), AP amplitude 85.9 and 87.1 mV
(p = 0.27), BH q = 0.55 and 0.54. The eEPSP amplitudes of 12 VEN-L and 12 VEN-S give p = 0.958 for
subtype in the two-way ANOVA (paper p = 0.9852). The paired comparisons with PCs come out as the
legend gives them (VEN-L p = 0.051 against 0.0500, VEN-S p = 0.037 against 0.0355). The test uses
the paper's per-cell tables, not the recordings, so it checks the statistics on the authors'
measurements, and the other features of Supplementary Table 8 were not read, as the criterion
says.

## fig3e-phase-peak-index (elsewhere: code repository table)

> VENs have lower AP phase-plane peak indices than every PC type.

The claims file points to the recordings of DANDI:001746, which were not streamed. The paper's
code repository holds the per-cell phase peak index by transcriptomic type
(`ephysio/data_ephy/PeakIdx_Fig5e.xlsx`), and this is used. Kruskal-Wallis test followed by Dunn's
test (Fig. 3e legend), unadjusted and Bonferroni-adjusted (the sheet's own statistics tab names
Bonferroni).

In [ ]:
# Per-cell phase peak index by transcriptomic type, from the paper's code repository
# (ephysio/data_ephy/PeakIdx_Fig5e.xlsx, Sheet1: one name column and one Peak_Index column per type).
# Rows are kept when the name is a cell ID (letters then digits, as s422'); this drops the mean,
# std and SE rows written under each column.
pk = pd.read_excel(io.BytesIO(nc.repo_file("ephysio/data_ephy/PeakIdx_Fig5e.xlsx")), sheet_name="Sheet1", header=None)
pgroups, dropped = {}, {}
for j in range(0, pk.shape[1], 3):
    ids = pk.iloc[1:, j].fillna("").astype(str)
    is_cell = ids.str.fullmatch(r"[A-Za-z]+\d+'?").values
    v = pd.to_numeric(pk.iloc[1:, j + 1], errors="coerce").values
    pgroups[pk.iloc[0, j]] = v[is_cell & ~np.isnan(v)]
    dropped[pk.iloc[0, j]] = int((~is_cell & ~np.isnan(v)).sum())
    if (is_cell & np.isnan(v)).any():
        print(f"{pk.iloc[0, j]}: cell IDs without a value:", ids[is_cell & np.isnan(v)].tolist())
legend_n = {"L23IT_HAPCAL1": 30, "ITrorb": 17, "L5IT": 119, "VEN": 22, "L5ET": 16, "L6PC": 13}
print(pd.DataFrame({"n": {k: len(v) for k, v in pgroups.items()}, "legend_n (Fig. 2d-f)": legend_n,
                    "summary rows dropped": dropped,
                    "median": {k: np.median(v) for k, v in pgroups.items()},
                    "range": {k: f"{v.min():.3f} to {v.max():.3f}" for k, v in pgroups.items()}}).to_string())
all_ids = pd.Series([i for j in range(0, pk.shape[1], 3) for i in pk.iloc[1:, j].fillna("").astype(str) if re.fullmatch(r"[A-Za-z]+\d+'?", i)])
print("Cell IDs listed in more than one type:", all_ids[all_ids.duplicated()].tolist())
kw = stats.kruskal(*pgroups.values())
print(f"Kruskal-Wallis H = {kw.statistic:.2f}, p = {kw.pvalue:.2g}")
# Dunn's test with the tie correction; p unadjusted and Bonferroni over the 15 pairs (the sheet's
# 'statistics' tab names Bonferroni-corrected Dunn's p values).
allv = np.concatenate(list(pgroups.values())); N = len(allv)
rk = stats.rankdata(allv)
_, cnt = np.unique(allv, return_counts=True)
tie = (cnt ** 3 - cnt).sum() / (12 * (N - 1))
mean_rank, k0 = {}, 0
for k, v in pgroups.items():
    mean_rank[k] = rk[k0:k0 + len(v)].mean(); k0 += len(v)
n_pairs = len(pgroups) * (len(pgroups) - 1) // 2
rows = []
for k in pgroups:
    if k == "VEN":
        continue
    se = np.sqrt((N * (N + 1) / 12 - tie) * (1 / len(pgroups["VEN"]) + 1 / len(pgroups[k])))
    z = (mean_rank["VEN"] - mean_rank[k]) / se
    p = 2 * stats.norm.sf(abs(z))
    rows.append({"VEN against": k, "z": z, "p": p, "p_bonferroni": min(1.0, p * n_pairs)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("VEN median lowest:", min(pgroups, key=lambda k: np.median(pgroups[k])) == "VEN")

**Result: reproduced.** The VEN median phase peak index (0.117, 29 cells) is the lowest of the six
types (0.223 to 0.338 for the others), Kruskal-Wallis p = 4.9e-08, and Dunn's test separates VENs
from every PC type (Bonferroni p = 1.5e-07 to 0.001; unadjusted 1e-08 to 6.8e-05). The sheet's
counts (30, 21, 121, 29, 17, 13) differ from those the legend refers to (Fig. 2d-f: 30, 17, 119,
22, 16, 13) in four of the six types, so the selection is not exactly the paper's. The test uses
the authors' per-cell values, so it checks the statistics, not the measure.

## fig3c-ais-distance-length (elsewhere: code repository table)

> In Pan-Nav stained sections the AIS of VENs lies farther from the soma than that of PCs, and VENs
> have a shorter AIS.

From `morpho/dataMorph/AIScmp_PCvsVEN_Fig5c.xlsx` (AIC_proj at commit 44cd8a6). The criterion's
test is a two-way ANOVA with slice or animal as the second factor (n = 108 VENs and 430 PCs, 23
slices, five animals, Fig. 3c legend).

In [ ]:
ais = pd.read_excel(io.BytesIO(nc.repo_file("morpho/dataMorph/AIScmp_PCvsVEN_Fig5c.xlsx")), header=None)
heads = {}
for j in ais.columns:
    hit = ais[j][ais[j].astype(str).str.contains("AIS_")]
    if len(hit):
        heads[j] = hit.iloc[0]
print("Columns:", heads)
vals = {h: pd.to_numeric(ais[j], errors="coerce").dropna() for j, h in heads.items()}
for h, v in vals.items():
    print(f"{h}: n = {len(v)}, median {v.median():.2f}, range {v.min():.2f} to {v.max():.2f}")
print("Other columns with values:", [j for j in ais.columns if j not in heads and ais[j].notna().sum() > 2])
print("Legend: n = 108 VENs and 430 PCs, 23 slices, five animals; the sheet has no slice or animal column.")
for a, b in [("VEN_AIS_D (um)", "PC_AIS_D (um)"), ("VEN_AIS_L (um)", "PC_AIS_L (um)")]:
    print(f"Description only, rank-sum {a} against {b}: p = {stats.mannwhitneyu(vals[a], vals[b]).pvalue:.3g}")

**Result: blocked (metadata insufficient).** The sheet holds AIS length and distance per cell,
by cell type, with no slice or animal, so neither reading of the two-way ANOVA can be formed. Its
counts (printed above) differ from the legend's 108 VENs and 430 PCs. The rank-sum tests printed
above describe the sheet and decide nothing.

## Claims not tested here

All are `data: elsewhere`; the attempt record gives the same reasons.

| Claim | Blocked by | Reason |
|---|---|---|
| model-sd-na-density, fig4i-peak-index-ais-distance, fig4g-model-ais-distance-ap-shape, model-ven-fires-more, ext7d-ais-length-no-effect | not_runnable | The repository at 44cd8a6 holds only MATLAB plotting scripts for the model (VEN_modeling/FigPlot); the model code is not in it, and the simulation outputs are in a six-part RAR archive on Zenodo (doi:10.5281/zenodo.18814764, about 11.8 GB) that cannot be read without downloading every part to disk. |
| fig5d-connectivity-totals, fig5-ven-sparse-connectivity | other | The criterion assigns the test to the claims file of DANDI:001751; it is not repeated here. |
| fig5b-psc-frequency, fig5c-psc-amplitude | other | The criterion assigns the test to the claims file of DANDI:001752; event detection on those recordings was not attempted. |
| scrna-classes, scrna-78-types | too_large | GEO GSE319557 holds FASTQ reads of 122,663 cells, whose alignment and clustering do not fit the runner; the processed data are in the multi-part RAR archive on Zenodo. |
| patchseq-mapping, ven-map-et-ct, fig6f-g-ven-subtype-mapping, fig6h-subtype-markers, fig7c-tau-rna-distance | too_large | They need the Patch-seq reads (GSE319369) mapped to the 10x reference, or the 10x expression data; the repository's mapping outputs give clusters as unnamed numbers. |
| ven-firing-rate-variability | methods_underspecified | The criterion blocks it unless the paper's Supplementary Tables 1 and 2 are supplied, as the text names no measure of spike-timing variability. The recordings, DANDI:001746, are also large (see the next row). |
| fig4ac-ap-width-amplitude, fig2df-l23-it-tau-width | too_large | DANDI:001746 (462 files, about 14.7 GB) would take about 1.9 to 3.2 hours to stream at the rates measured in this attempt's runs (0.33 GB in 150 to 256 s), beyond the notebook's 60 minutes. No repository table holds AP width and amplitude of 281 VENs and 306 PCs; the repository's Patch-seq feature table (Ins_patchseq_ephs.xlsx) gives cluster numbers, not the subclass names fig2df needs. |
| fig3ab-ven-axon-from-basal-dendrite, fig6-ven-subtype-morphology, ext7i-basal-dendrite-diameter, fig3f-ais-distance-types | other | The morphologies are on NeuroMorpho.org (doi:10.13021/y4be-0p18); tracing each reconstruction was not attempted in the time of the run. fig3f also needs the Patch-seq subclasses. |
| fig4j-eepsp-latency | other | sumData_StimWM.xlsx, the table the claims file names, holds amplitudes and no latency, and the paper does not say which dandiset holds the raw eEPSP recordings. |

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| na-cell-table-link | with labels | partially reproduced |
| fig4d-sodium-current-lower | dataset (types from the link) | blocked: methods underspecified |
| model-na-ratio | dataset | blocked: methods underspecified (depends on fig4d) |
| na-patch-capacitance | dataset | blocked: data missing |
| fig8a-metabolite-gradient | elsewhere (MetaboLights) | not reproduced |
| fig8de-glycolytic-bias | elsewhere (MetaboLights) | partially reproduced |
| fig4j-eepsp-amplitude | elsewhere (code repository table) | reproduced |
| ven-subtypes-similar-physiology | elsewhere (code repository tables) | reproduced |
| fig3e-phase-peak-index | elsewhere (code repository table) | reproduced |
| fig3c-ais-distance-length | elsewhere (code repository table) | blocked: metadata insufficient |
| 24 other claims on data elsewhere | elsewhere | blocked (see "Claims not tested here") |

**How the analysis differs from the paper's.** The paper does not say how the maximal Na+ current
was measured from the traces; the claims file's eight readings were run, and they disagree on
significance. Those readings take one sample of an unfiltered trace, and under the code's example
reading (a2 b1 c1) in 10 of 35 cells (6 of the 14 VENs) that sample is a brief transient, an
artefact, not a Na+ current; 9 of the 10 are in session AS. With a 2.9 kHz low-pass filter or a
0 to 3 ms window, every baseline-subtracted variant finds VENs significantly lower; these variants decide
nothing, and whether the criterion should include one is asked of a reader. No measure tried
matches the code table's values to the criterion's tolerance; a peak taken after smoothing the
traces, a step the paper does not describe, comes closest (26 of 35 within 10%, at a width fitted
to the table). Current density could not be computed, since the files hold no capacitance. The
metabolomics counts use the deposited values with no transform, which the Methods do not name.
The eEPSP, VEN subtype and phase peak index claims are tested on the authors' per-cell tables,
not on recordings.

**What was found about the dataset.** The 35 files match the paper's code table one to one by
file name and cell number, with 14 VEN and 21 PC by that table, but the files carry no cell type,
so the dataset alone cannot be split into VENs and PCs. No file holds the patch capacitance or
any amplifier compensation value. The traces keep holding currents of -131 to +92 pA, so a
baseline must be subtracted before a peak is taken; one file (HS018) has a positive holding
current and noise near 57 pA. Files hold 28 or 34 sweeps (to +55 or +85 mV) and 1 to 10
repetitions, where the Methods describe steps to 0 mV. The series' starting times (0 to 3 s) carry
no order, and every file gives the placeholder session start 2018-01-01. Brief biphasic
transients (about 0.1 ms) appear in the averaged traces of at least 10 files.